# Lab 16-10 · Bits, Noise and Limit Cycles

Five parts, each built on `quant.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math, sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import quant as q
sys.path.insert(0, "../16-06-lab")
import firdesign as fd

## Part 1 · 6 dB per bit

*Predict:* the slope and intercept of SQNR against bits.

In [2]:
Ns = np.arange(8, 21)
s = np.array([q.sine_sqnr(int(N), amp=(2 ** (N - 1) - 1) / 2 ** (N - 1)) for N in Ns])
slope, icpt = np.polyfit(Ns, s, 1)
print(f"slope {slope:.3f} dB/bit, intercept {icpt:.2f} dB; largest deviation from 6.02N + 1.76: {np.max(np.abs(s - (6.02 * Ns + 1.76))):.3f} dB")
assert abs(slope - 6.02) < 0.005 and abs(icpt - 1.81) < 0.005 and np.max(np.abs(s - (6.02 * Ns + 1.76))) < 0.05

slope 6.018 dB/bit, intercept 1.81 dB; largest deviation from 6.02N + 1.76: 0.049 dB


## Part 2 · Loading a Gaussian signal

*Predict:* the best loading factor at 12 and at 16 bits.

In [3]:
def sqnr_formula(L, B):
    s = 1 / L
    D = 2.0 ** -(B - 1)
    clip = 2 * s * s * ((1 + L * L) * norm.sf(L) - L * norm.pdf(L))
    return 10 * math.log10(s * s / (D * D / 12 + clip))
# Simulation is trustworthy only where clipping events are plentiful: at L = 4 a million samples clip about 63
# times, at L = 5 about once. Compare simulation and formula for L = 3 to 4, then use the formula beyond.
for B in (12, 16):
    for L in (3.0, 3.5, 4.0):
        sim, form = q.gaussian_sqnr(B, L), sqnr_formula(L, B)
        print(f"{B} bits, L = {L}: simulated {sim:.2f} dB, formula {form:.2f} dB, expected clipped samples per million {2e6 * norm.sf(L):.0f}")
        assert abs(sim - form) < 0.3
Ls = np.linspace(3, 7, 401)
for B in (12, 16):
    form = np.array([sqnr_formula(L, B) for L in Ls])
    print(f"{B} bits: best L = {Ls[np.argmax(form)]:.2f}, {form.max():.1f} dB")
    plt.plot(Ls, form, label=f"{B} bits")
plt.xlabel("loading L (full scale / sigma)"); plt.ylabel("SQNR (dB)"); plt.legend(); plt.show()
assert abs(q.gaussian_sqnr(12, 4.0) - 51.7) < 0.05

12 bits, L = 3.0: simulated 34.05 dB, formula 33.90 dB, expected clipped samples per million 2700
12 bits, L = 3.5: simulated 42.55 dB, formula 42.51 dB, expected clipped samples per million 465
12 bits, L = 4.0: simulated 51.66 dB, formula 51.87 dB, expected clipped samples per million 63
16 bits, L = 3.0: simulated 34.06 dB, formula 33.91 dB, expected clipped samples per million 2700
16 bits, L = 3.5: simulated 42.59 dB, formula 42.53 dB, expected clipped samples per million 465
16 bits, L = 4.0: simulated 51.88 dB, formula 52.09 dB, expected clipped samples per million 63


12 bits: best L = 5.01, 62.7 dB


16 bits: best L = 5.94, 85.4 dB


## Part 3 · Distortion and dither

*Predict:* the level of the third harmonic of a 1.5 LSB sine, undithered.

In [4]:
N, k0 = 4096, 67
x = 1.5 * np.sin(2 * np.pi * k0 * np.arange(N) / N + 0.3)
xq = np.floor(x + 0.5)
h = q.harmonic_levels_db(xq, k0, 7)
print("undithered harmonics 2..7 (dBc):", [round(v, 1) if v > -300 else None for v in h], " error variance", round(float(np.var(xq - x)), 3))
assert abs(h[1] + 14.6) < 0.05
rng = np.random.default_rng(5)
xd = np.floor(x + q.tpdf_dither(N, rng) + 0.5)
hd = q.harmonic_levels_db(xd, k0, 7)
print("dithered harmonic bins (dBc):", [round(v, 1) for v in hd], " error variance", round(float(np.var(xd - x)), 3))
assert max(hd) < -30 and abs(np.var(xd - x) - 0.25) < 0.02

undithered harmonics 2..7 (dBc): [None, -14.6, None, -31.3, None, -19.2]  error variance 0.108
dithered harmonic bins (dBc): [-42.3, -37.3, -45.5, -38.3, -38.6, -38.6]  error variance 0.254


## Part 4 · How many coefficient bits?

*Predict:* the fewest fractional bits that keep the 137-tap filter at 60 dB.

In [5]:
fs = 48000
wp, ws = 2 * np.pi * 4000 / fs, 2 * np.pi * 5000 / fs
dp, ds = fd.deltas(0.1, 60)
h = fd.remez_lowpass(137, wp, ws, dp, ds)
att = {}
for F in range(10, 19):
    m = fd.measure(q.quantize(h, F), wp, ws)
    att[F] = m[3]
    print(f"{F} fractional bits: attenuation {m[3]:.2f} dB, passband ripple {m[2]:.4f} dB")
assert abs(att[12] - 51.85) < 0.005 and abs(att[18] - 60.17) < 0.005 and min(F for F in att if att[F] >= 60) == 18

10 fractional bits: attenuation 39.77 dB, passband ripple 0.1271 dB


11 fractional bits: attenuation 44.81 dB, passband ripple 0.1357 dB


12 fractional bits: attenuation 51.85 dB, passband ripple 0.1254 dB


13 fractional bits: attenuation 55.06 dB, passband ripple 0.1079 dB


14 fractional bits: attenuation 56.63 dB, passband ripple 0.1001 dB


15 fractional bits: attenuation 58.71 dB, passband ripple 0.0982 dB


16 fractional bits: attenuation 59.56 dB, passband ripple 0.0968 dB


17 fractional bits: attenuation 59.92 dB, passband ripple 0.0967 dB


18 fractional bits: attenuation 60.17 dB, passband ripple 0.0962 dB


## Part 5 · Limit cycles

*Predict:* where does $y = Q(0.9y)$ stop if it starts at 20 LSB?

In [6]:
r1 = q.limit_cycle_first_order(0.9, 20, "round")
r2 = q.limit_cycle_first_order(-0.9, 20, "round")
r3 = q.limit_cycle_first_order(0.9, 20, "zero")
print("a = 0.9:", r1[:14], "...", r1[-3:])
print("a = -0.9 tail:", r2[-4:], " magnitude truncation tail:", r3[-3:])
assert r1[-1] == 5 and set(map(abs, r2[-4:])) == {4} and r3[-1] == 0
w = q.second_order_zero_input(-1.6, 0.9, 0.9, -0.9, "wrap", n=1000)
s = q.second_order_zero_input(-1.6, 0.9, 0.9, -0.9, "saturate", n=1000)
print("wrap-around tail peak:", max(map(abs, w[-50:])), " saturation tail peak:", max(map(abs, s[-50:])))
assert max(map(abs, w[-50:])) == 31039 and max(map(abs, s[-50:])) == 3
plt.figure(figsize=(7, 3)); plt.plot(w, lw=0.7, label="wrap-around"); plt.plot(s, lw=0.7, label="saturation")
plt.legend(); plt.xlabel("sample"); plt.ylabel("output (LSB)"); plt.show()

a = 0.9: [18, 16, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 5, 5] ... [5, 5, 5]
a = -0.9 tail: [-4, 4, -4, 4]  magnitude truncation tail: [0, 0, 0]
wrap-around tail peak: 31039  saturation tail peak: 3
